# ==============================
# Mushroom Classification - Full Multi-Model Optuna Pipeline
# ==============================

In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt

In [ ]:

# ===============================
# 1. Load Data
# ===============================
train = pd.read_csv("/kaggle/input/mushroom-classification/train.csv")
test = pd.read_csv("/kaggle/input/mushroom-classification/test.csv")
sample_submission = pd.read_csv("/kaggle/input/mushroom-classification/sample_submission.csv")

In [ ]:

# ===============================
# 2. Drop High Null Columns
# ===============================
drop_cols = ["stem-root", "stem-surface", "veil-type", "veil-color", "spore-print-color"]
X = train.drop(columns=["id", "class"] + drop_cols)
X_test = test.drop(columns=["id"] + drop_cols)

# Encode target
le = LabelEncoder()
y = le.fit_transform(train["class"])  # 'e'->0, 'p'->1

In [ ]:

# ===============================
# 3. Train/Validation Split
# ===============================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ===============================
# 4. Define Preprocessing
# ===============================
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["float64", "int64"]).columns

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", categorical_transformer, categorical_cols),
    ("num", numerical_transformer, numerical_cols)
])

In [ ]:
# ===============================
# 5. Optuna Objective Function
# ===============================
def objective(trial):

    model_name = trial.suggest_categorical(
        "model",
        ["LogisticRegression", "RandomForest", "XGBoost", "SVM",
         "GradientBoosting", "LightGBM", "CatBoost", "KNN"]
    )

    # ---------------- Logistic Regression ----------------
    if model_name == "LogisticRegression":
        C = trial.suggest_float("lr_C", 0.1, 100, log=True)
        solver = trial.suggest_categorical("lr_solver", ["liblinear", "lbfgs", "saga", "newton-cg"])
        penalty = None
        if solver in ["lbfgs", "newton-cg"]:
            penalty = trial.suggest_categorical("lr_penalty_lbfgs_nc", ["l2", None])
        elif solver == "liblinear":
            penalty = trial.suggest_categorical("lr_penalty_liblinear", ["l1", "l2"])
        else:  # saga
            penalty = trial.suggest_categorical("lr_penalty_saga", ["l1", "l2", "elasticnet", None])
        l1_ratio = trial.suggest_float("lr_l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None

        model = LogisticRegression(
            C=C, solver=solver, penalty=penalty, l1_ratio=l1_ratio,
            max_iter=5000, random_state=42
        )

    # ---------------- Random Forest ----------------
    elif model_name == "RandomForest":
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 50, 300),
            max_depth=trial.suggest_int("rf_max_depth", 3, 20),
            min_samples_split=trial.suggest_int("rf_min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 1, 10),
            bootstrap=trial.suggest_categorical("rf_bootstrap", [True, False]),
            random_state=42, n_jobs=1
        )

    # ---------------- XGBoost ----------------
    elif model_name == "XGBoost":
        model = XGBClassifier(
            learning_rate=trial.suggest_float("xgb_learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("xgb_max_depth", 2, 12),
            n_estimators=trial.suggest_int("xgb_n_estimators", 50, 500),
            subsample=trial.suggest_float("xgb_subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("xgb_colsample_bytree", 0.5, 1.0),
            gamma=trial.suggest_float("xgb_gamma", 0, 5),
            min_child_weight=trial.suggest_int("xgb_min_child_weight", 1, 10),
            reg_alpha=trial.suggest_float("xgb_reg_alpha", 0.0, 5.0),
            reg_lambda=trial.suggest_float("xgb_reg_lambda", 0.0, 5.0),
            random_state=42, use_label_encoder=False, eval_metric="logloss", n_jobs=1
        )

    # ---------------- SVM ----------------
    elif model_name == "SVM":
        model = SVC(
            C=trial.suggest_float("svm_C", 0.1, 100, log=True),
            kernel=trial.suggest_categorical("svm_kernel", ["linear", "rbf", "poly", "sigmoid"]),
            gamma=trial.suggest_categorical("svm_gamma", ["scale", "auto"]),
            probability=True, random_state=42
        )

    # ---------------- Gradient Boosting ----------------
    elif model_name == "GradientBoosting":
        model = GradientBoostingClassifier(
            n_estimators=trial.suggest_int("gb_n_estimators", 50, 500),
            learning_rate=trial.suggest_float("gb_learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("gb_max_depth", 2, 10),
            subsample=trial.suggest_float("gb_subsample", 0.5, 1.0),
            min_samples_split=trial.suggest_int("gb_min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("gb_min_samples_leaf", 1, 20),
            random_state=42
        )

    # ---------------- LightGBM ----------------
    elif model_name == "LightGBM":
        model = LGBMClassifier(
            n_estimators=trial.suggest_int("lgb_n_estimators", 50, 500),
            learning_rate=trial.suggest_float("lgb_learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("lgb_max_depth", -1, 20),
            num_leaves=trial.suggest_int("lgb_num_leaves", 20, 300),
            subsample=trial.suggest_float("lgb_subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("lgb_colsample_bytree", 0.5, 1.0),
            random_state=42, n_jobs=1
        )

    # ---------------- CatBoost ----------------
    elif model_name == "CatBoost":
        model = CatBoostClassifier(
            iterations=trial.suggest_int("cat_iterations", 100, 500),
            depth=trial.suggest_int("cat_depth", 3, 10),
            learning_rate=trial.suggest_float("cat_learning_rate", 0.01, 0.3, log=True),
            l2_leaf_reg=trial.suggest_float("cat_l2_leaf_reg", 1.0, 10.0),
            verbose=0, random_state=42
        )

    # ---------------- KNN ----------------
    elif model_name == "KNN":
        model = KNeighborsClassifier(
            n_neighbors=trial.suggest_int("knn_n_neighbors", 3, 30),
            weights=trial.suggest_categorical("knn_weights", ["uniform", "distance"]),
            p=trial.suggest_int("knn_p", 1, 2)
        )

    # ---------------- Pipeline ----------------
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    # Use n_jobs=1 to avoid Windows parallelism errors
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring="accuracy", n_jobs=1)
    return scores.mean()


In [ ]:
# ===============================
# 6. Run Optuna Study
# ===============================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # Increase trials for better results

print("Best Params:", study.best_params)
best_model_name = study.best_params["model"]


In [ ]:
# ===============================
# 7. Train Best Model on Full Training Data
# ===============================
# Recreate best model with params (you can extract these from study.best_params)
# Here we only demo with RandomForest as example (adapt as per best_model_name)

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=study.best_params.get("rf_n_estimators", 100),
        max_depth=study.best_params.get("rf_max_depth", None),
        min_samples_split=study.best_params.get("rf_min_samples_split", 2),
        min_samples_leaf=study.best_params.get("rf_min_samples_leaf", 1),
        bootstrap=study.best_params.get("rf_bootstrap", True),
        random_state=42,
        n_jobs=-1
    ))
])




In [ ]:
best_model.fit(X, y)  # Full training



In [ ]:
# ===============================
# 8. Predict on Test Set
# ===============================

# Validate on hold-out set
y_valid_pred = best_model.predict(X_valid)
val_acc = accuracy_score(y_valid, y_valid_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

In [ ]:
# ===============================
# 8. Predict on Test + Submission
# ===============================
y_pred = best_model.predict(X_test)
y_pred_labels = le.inverse_transform(y_pred)  # back to 'e' / 'p'

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "class": y_pred_labels
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv file created with labels (e/p)")

# Using ROC-Curve

In [ ]:
# Probabilities & ROC Curve

y_proba = best_model.predict_proba(X_valid)[:, 1]  # probability for class 1

In [ ]:
roc_auc = roc_auc_score(y_valid, y_proba)
print("ROC-AUC Score:", roc_auc)

In [ ]:
# Compute ROC curve

fpr, tpr, thresholds = roc_curve(y_valid, y_proba)

# Best Threshold using Youden's J statistic
j_scores = tpr - fpr
j_best_idx = j_scores.argmax()
best_threshold = thresholds[j_best_idx]
print("Best Threshold:", best_threshold)

# ===============================
# Plot ROC curve with Best Threshold
# ===============================
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"{best_model_name} (AUC = {roc_auc:.3f})")
#plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

# Mark best threshold
plt.scatter(fpr[j_best_idx], tpr[j_best_idx], color='red', s=100, label=f"Best Threshold = {best_threshold:.3f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve with Best Threshold")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Predictions using Best Threshold
y_pred_best = (y_proba >= best_threshold).astype(int)

In [ ]:
y_pred_best_labels = le.inverse_transform(y_pred_best)

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "class": y_pred_best_labels
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv file created with labels (e/p)")